In [ ]:
import subprocess, sys

for pkg in ["openai", "sentence-transformers", "faiss-cpu", "tabulate", "tqdm", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

In [ ]:
import os, sys

REPO_URL    = "https://github.com/vudinhminh08/NLP-project-master-study.git"
REPO_BRANCH = "master"
PROJECT_DIR = "/kaggle/working/absa-project"

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
for d in ["data", "outputs/results", "outputs/llm_cache", "outputs/eda"]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, os.path.join(PROJECT_DIR, "code", "llm_rag"))
sys.path.insert(0, os.path.join(PROJECT_DIR, "code", "data_processing"))

In [ ]:
import pandas as pd

if not os.path.exists("data/train_preprocessed.csv"):
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/

train_df = pd.read_csv("data/train_preprocessed.csv")
test_df  = pd.read_csv("data/test_preprocessed.csv")

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    _secret_client = UserSecretsClient()
except Exception:
    _secret_client = None

def get_secret_or_env(name: str, default: str = "") -> str:
    if os.environ.get(name):
        return os.environ[name]
    if _secret_client is not None:
        try:
            value = _secret_client.get_secret(name)
            if value:
                return value
        except Exception:
            pass
    return default

OPENAI_API_KEY = get_secret_or_env("OPENAI_API_KEY")
GEMINI_API_KEY = get_secret_or_env("GEMINI_API_KEY")
OLLAMA_API_KEY = get_secret_or_env("OLLAMA_API_KEY")
OLLAMA_BASE_URL = get_secret_or_env("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = get_secret_or_env("OLLAMA_MODEL", "llama3.1:8b")
OLLAMA_ENABLE = get_secret_or_env("OLLAMA_ENABLE", "0")

os.environ["OLLAMA_BASE_URL"] = OLLAMA_BASE_URL
os.environ["OLLAMA_MODEL"] = OLLAMA_MODEL
os.environ["OLLAMA_ENABLE"] = OLLAMA_ENABLE

api_keys = {}
if OPENAI_API_KEY:
    api_keys["openai"] = OPENAI_API_KEY
if GEMINI_API_KEY:
    api_keys["gemini"] = GEMINI_API_KEY
if OLLAMA_API_KEY:
    api_keys["ollama"] = OLLAMA_API_KEY

providers = []
if "openai" in api_keys:
    providers.append("openai")
if "gemini" in api_keys:
    providers.append("gemini")
if OLLAMA_ENABLE == "1":
    providers.append("ollama")

if not providers:
    raise ValueError(
        "No provider configured. Set OPENAI_API_KEY/GEMINI_API_KEY or enable Ollama with OLLAMA_ENABLE=1."
)

models = {
    "openai": "gpt-4o-mini",
    "gemini": "gemini-2.0-flash",
    "ollama": OLLAMA_MODEL,
}

print("Providers:", providers)
print("Ollama enabled:", OLLAMA_ENABLE == "1")
print("Ollama base URL:", OLLAMA_BASE_URL)

In [ ]:
from icl_predictor import run_icl_ablation

# ICL supports openai/gemini. Ollama is evaluated in RAG cell below.
icl_providers = [p for p in providers if p in ["openai", "gemini"]]
if not icl_providers:
    print("Skip ICL: no openai/gemini key configured.")
    icl_results = {}
else:
    icl_results = run_icl_ablation(
        test_df=test_df,
        train_df=train_df,
        providers=icl_providers,
        k_values=[2, 4, 8],
        api_keys=api_keys,
        models=models,
        results_dir="outputs/results",
        max_samples=None,
    )

In [ ]:
from rag_predictor import run_rag_ablation

rag_results = run_rag_ablation(
    test_df=test_df,
    train_df=train_df,
    providers=providers,
    k_values=[2, 4, 8],
    api_keys=api_keys,
    models=models,
    results_dir="outputs/results",
    max_samples=None,
)

In [ ]:
from compare_results import generate_comparison_table
print(generate_comparison_table(save_path="outputs/results/final_four_direction_comparison.md"))

In [ ]:
import json, os, matplotlib.pyplot as plt

k_values = [2, 4, 8]
provider = providers[0]

icl_f1s, rag_f1s = [], []
for k in k_values:
    icl_path = f"outputs/results/icl_{provider}_k{k}_metrics.json"
    rag_path = f"outputs/results/rag_{provider}_k{k}_metrics.json"

    if os.path.exists(icl_path):
        icl = json.load(open(icl_path))
        icl_f1s.append(icl["macro_combined_f1"] )
    else:
        icl_f1s.append(None)

    if os.path.exists(rag_path):
        rag = json.load(open(rag_path))
        rag_f1s.append(rag["macro_combined_f1"] )
    else:
        rag_f1s.append(None)

PHOBERT_F1 = 0.5543
SVM_F1     = 0.3173

fig, ax = plt.subplots(figsize=(8, 5))
if any(v is not None for v in icl_f1s):
    ax.plot(k_values, [v if v is not None else float("nan") for v in icl_f1s], "o-", color="steelblue", label="ICL (random)", lw=2)
ax.plot(k_values, [v if v is not None else float("nan") for v in rag_f1s], "o-", color="darkorange", label="RAG 2-stage (ACD->SPC)", lw=2)
ax.axhline(y=PHOBERT_F1, color="green", ls="--", label=f"PhoBERT ({PHOBERT_F1:.4f})", alpha=0.7)
ax.axhline(y=SVM_F1,     color="gray",  ls=":",  label=f"SVM ({SVM_F1:.4f})", alpha=0.7)
ax.set(title=f"{provider} — ICL vs RAG 2-stage (k ablation)", xlabel="k (so examples)", ylabel="Combined F1")
ax.set_xticks(k_values)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/eda/llm_rag_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/llm_rag_results", "zip", "outputs/results")